In [0]:
dbutils.notebook.run(
    "/Users/aadityrathore0344@gmail.com/financial-market-intelligence/Set_UP/store_key",
    timeout_seconds=60
)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load silver data
print("📊 Loading silver layer data...\n")

# 1. Stocks data
silver_stocks = spark.table("financial_market.silver.stocks")
print(f"Stocks: {silver_stocks.count():,} records")
print(f"  Date range: {silver_stocks.agg(F.min('date')).collect()[0][0]} to {silver_stocks.agg(F.max('date')).collect()[0][0]}")
print(f"  Tickers: {silver_stocks.select('ticker').distinct().count()}")

# 2. Crypto data
silver_crypto = spark.table("financial_market.silver.crypto")
print(f"\nCrypto: {silver_crypto.count():,} records")
print(f"  Date range: {silver_crypto.agg(F.min('fetch_date')).collect()[0][0]} to {silver_crypto.agg(F.max('fetch_date')).collect()[0][0]}")
print(f"  Symbols: {silver_crypto.select('symbol').distinct().count()}")

# 3. Forex data
silver_forex = spark.table("financial_market.silver.forex")
print(f"\nForex: {silver_forex.count():,} records")
print(f"  Date range: {silver_forex.agg(F.min('rate_date')).collect()[0][0]} to {silver_forex.agg(F.max('rate_date')).collect()[0][0]}")
print(f"  Currency pairs: {silver_forex.select('pair').distinct().count()}")

print("\n✅ All data loaded successfully!")

In [0]:
# ============================
# DATA PREPARATION
# ============================

print("🛠️ Preparing data for correlation analysis...\n")

# 1. Prepare stocks data - use daily_change_pct as return metric
stocks_prep = silver_stocks.select(
    F.col("date"),
    F.col("ticker"),
    F.col("daily_change_pct").alias("return_pct")
)

print(f"Stocks prepared: {stocks_prep.count():,} records")

# 2. Prepare crypto data - use price_change_24h_pct
# Rename fetch_date to date for consistency
crypto_prep = silver_crypto.select(
    F.col("fetch_date").alias("date"),
    F.col("symbol").alias("ticker"),
    F.col("price_change_24h_pct").alias("return_pct")
)

print(f"Crypto prepared: {crypto_prep.count():,} records")

# 3. Prepare forex data - calculate daily change percentage
# First, add previous day rate using window function
forex_window = Window.partitionBy("pair").orderBy("rate_date")

forex_with_prev = silver_forex \
    .withColumn("prev_rate", F.lag("exchange_rate").over(forex_window)) \
    .withColumn("return_pct", 
                F.when(F.col("prev_rate").isNotNull(),
                      F.round((F.col("exchange_rate") - F.col("prev_rate")) / F.col("prev_rate") * 100, 4))
                .otherwise(0))

forex_prep = forex_with_prev.select(
    F.col("rate_date").alias("date"),
    F.col("pair").alias("ticker"),
    F.col("return_pct")
).filter(F.col("return_pct") != 0)  # Remove first day with no previous rate

print(f"Forex prepared: {forex_prep.count():,} records")

# 4. Add asset type labels
stocks_labeled = stocks_prep.withColumn("asset_type", F.lit("STOCK"))
crypto_labeled = crypto_prep.withColumn("asset_type", F.lit("CRYPTO"))
forex_labeled = forex_prep.withColumn("asset_type", F.lit("FOREX"))

# 5. Union all datasets
all_assets = stocks_labeled.union(crypto_labeled).union(forex_labeled)

print(f"\nTotal combined records: {all_assets.count():,}")
print(f"Date range: {all_assets.agg(F.min('date')).collect()[0][0]} to {all_assets.agg(F.max('date')).collect()[0][0]}")
print(f"Unique assets: {all_assets.select('ticker').distinct().count()}")

print("\n📋 Sample of prepared data:")
display(all_assets.orderBy(F.col("date").desc()).limit(10))

In [0]:
# ============================
# CORRELATION CALCULATION
# ============================

import pandas as pd

print("📊 Calculating cross-asset correlations...\n")

# Note: Forex has 0 records, Crypto has only 1 date
# Focus on stock correlations and include crypto where available

# Convert to Pandas for easier manipulation (avoiding RDD operations)
print("Converting asset data to Pandas...")
all_assets_pd = all_assets.toPandas()

print(f"Total records: {len(all_assets_pd)}")
print(f"\nAssets by type:")
for asset_type in ['STOCK', 'CRYPTO', 'FOREX']:
    count = all_assets_pd[all_assets_pd['asset_type'] == asset_type]['ticker'].nunique()
    records = len(all_assets_pd[all_assets_pd['asset_type'] == asset_type])
    print(f"  {asset_type}: {count} assets, {records} records")

# Get top 5 stocks by data availability
top_stocks = all_assets_pd[all_assets_pd['asset_type'] == 'STOCK'].groupby('ticker').size() \
    .nlargest(5).index.tolist()

# Get all crypto
top_crypto = all_assets_pd[all_assets_pd['asset_type'] == 'CRYPTO']['ticker'].unique().tolist()

print(f"\nSelected assets:")
print(f"  Stocks: {top_stocks}")
print(f"  Crypto: {top_crypto}")

# Filter to selected assets
selected_tickers = top_stocks + top_crypto
filtered_data = all_assets_pd[all_assets_pd['ticker'].isin(selected_tickers)]

print(f"\nFiltered data: {len(filtered_data):,} records")
print(f"Unique assets: {filtered_data['ticker'].nunique()}")

# Pivot: dates as rows, tickers as columns, return_pct as values
print("\nPivoting data...")
pivot_df = filtered_data.pivot(index='date', columns='ticker', values='return_pct')

print(f"Pivoted data shape: {pivot_df.shape}")
print(f"  Dates: {pivot_df.shape[0]}")
print(f"  Assets: {pivot_df.shape[1]}")
print(f"  Missing values: {pivot_df.isnull().sum().sum()}")

# Fill missing values with 0 (no change)
pivot_df_filled = pivot_df.fillna(0)

# Calculate correlation matrix
print("\nCalculating correlation matrix...")
corr_matrix = pivot_df_filled.corr()

print(f"Correlation matrix shape: {corr_matrix.shape}")
print("\n📋 Correlation Matrix (first 5x5):")
print(corr_matrix.iloc[:5, :5].round(3))

In [0]:
# ============================
# IDENTIFY STRONG CORRELATIONS
# ============================

print("🔍 Identifying strong correlations...\n")

# Convert correlation matrix to long format for analysis
corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):  # Upper triangle only
        asset1 = corr_matrix.columns[i]
        asset2 = corr_matrix.columns[j]
        corr_value = corr_matrix.iloc[i, j]
        
        # Determine asset types
        asset1_type = "STOCK" if asset1 in top_stocks else ("CRYPTO" if asset1 in top_crypto else "FOREX")
        asset2_type = "STOCK" if asset2 in top_stocks else ("CRYPTO" if asset2 in top_crypto else "FOREX")
        
        corr_pairs.append({
            'asset1': asset1,
            'asset2': asset2,
            'asset1_type': asset1_type,
            'asset2_type': asset2_type,
            'correlation': corr_value,
            'abs_correlation': abs(corr_value)
        })

corr_pairs_df = pd.DataFrame(corr_pairs)

print(f"Total correlation pairs: {len(corr_pairs_df)}")

# Top positive correlations
print("\n🔼 TOP 10 POSITIVE CORRELATIONS:")
top_positive = corr_pairs_df.nlargest(10, 'correlation')
for idx, row in top_positive.iterrows():
    print(f"  {row['asset1']} ({row['asset1_type']}) <-> {row['asset2']} ({row['asset2_type']}): {row['correlation']:.4f}")

# Top negative correlations
print("\n🔽 TOP 10 NEGATIVE CORRELATIONS:")
top_negative = corr_pairs_df.nsmallest(10, 'correlation')
for idx, row in top_negative.iterrows():
    print(f"  {row['asset1']} ({row['asset1_type']}) <-> {row['asset2']} ({row['asset2_type']}): {row['correlation']:.4f}")

# Strongest correlations by asset type pairs
print("\n🔀 CROSS-ASSET TYPE CORRELATIONS:")
for type1 in ['STOCK', 'CRYPTO', 'FOREX']:
    for type2 in ['STOCK', 'CRYPTO', 'FOREX']:
        if type1 < type2:  # Avoid duplicates
            cross_corr = corr_pairs_df[
                ((corr_pairs_df['asset1_type'] == type1) & (corr_pairs_df['asset2_type'] == type2)) |
                ((corr_pairs_df['asset1_type'] == type2) & (corr_pairs_df['asset2_type'] == type1))
            ]
            if len(cross_corr) > 0:
                avg_corr = cross_corr['correlation'].mean()
                max_corr = cross_corr['correlation'].max()
                min_corr = cross_corr['correlation'].min()
                print(f"\n  {type1} <-> {type2}:")
                print(f"    Average: {avg_corr:.4f}")
                print(f"    Max: {max_corr:.4f}")
                print(f"    Min: {min_corr:.4f}")
                print(f"    Pairs: {len(cross_corr)}")

# Display top correlations as table
print("\n📋 Top Correlations Table:")
display(spark.createDataFrame(top_positive[['asset1', 'asset1_type', 'asset2', 'asset2_type', 'correlation']]))

In [0]:
# ============================
# SAVE TO GOLD LAYER
# ============================

print("📦 Saving correlation data to gold layer...\n")

# Ensure gold schema exists
spark.sql("CREATE SCHEMA IF NOT EXISTS financial_market.gold")

# 1. Save correlation matrix in long format
print("1. Saving correlation matrix...")
corr_matrix_long = []
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        asset1 = corr_matrix.columns[i]
        asset2 = corr_matrix.columns[j]
        corr_value = corr_matrix.iloc[i, j]
        
        # Determine asset types
        asset1_type = "STOCK" if asset1 in top_stocks else ("CRYPTO" if asset1 in top_crypto else "FOREX")
        asset2_type = "STOCK" if asset2 in top_stocks else ("CRYPTO" if asset2 in top_crypto else "FOREX")
        
        corr_matrix_long.append({
            'asset1': asset1,
            'asset2': asset2,
            'asset1_type': asset1_type,
            'asset2_type': asset2_type,
            'correlation': float(corr_value),
            'processed_timestamp': pd.Timestamp.now()
        })

corr_matrix_df = spark.createDataFrame(pd.DataFrame(corr_matrix_long))

corr_matrix_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("financial_market.gold.asset_correlation_matrix")

print("✅ Saved to financial_market.gold.asset_correlation_matrix")

# 2. Save top correlation pairs
print("\n2. Saving correlation pairs...")
corr_pairs_spark = spark.createDataFrame(corr_pairs_df)

corr_pairs_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("financial_market.gold.asset_correlation_pairs")

print("✅ Saved to financial_market.gold.asset_correlation_pairs")

# 3. Create and save correlation summary by asset type
print("\n3. Creating correlation summary by asset type...")
summary_data = []
for type1 in ['STOCK', 'CRYPTO', 'FOREX']:
    for type2 in ['STOCK', 'CRYPTO', 'FOREX']:
        cross_corr = corr_pairs_df[
            ((corr_pairs_df['asset1_type'] == type1) & (corr_pairs_df['asset2_type'] == type2)) |
            ((corr_pairs_df['asset1_type'] == type2) & (corr_pairs_df['asset2_type'] == type1))
        ]
        if len(cross_corr) > 0:
            summary_data.append({
                'asset_type_1': type1,
                'asset_type_2': type2,
                'avg_correlation': float(cross_corr['correlation'].mean()),
                'max_correlation': float(cross_corr['correlation'].max()),
                'min_correlation': float(cross_corr['correlation'].min()),
                'median_correlation': float(cross_corr['correlation'].median()),
                'std_correlation': float(cross_corr['correlation'].std()),
                'pair_count': len(cross_corr),
                'processed_timestamp': pd.Timestamp.now()
            })

summary_df = spark.createDataFrame(pd.DataFrame(summary_data))

summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("financial_market.gold.asset_correlation_summary")

print("✅ Saved to financial_market.gold.asset_correlation_summary")

print("\n✅ All correlation data saved successfully!")
print("\n📊 Gold Layer Tables:")
print("  - financial_market.gold.asset_correlation_matrix")
print("  - financial_market.gold.asset_correlation_pairs")
print("  - financial_market.gold.asset_correlation_summary")

In [0]:
# ============================
# VISUALIZATION
# ============================

import matplotlib.pyplot as plt
import seaborn as sns

print("📊 Creating correlation heatmap...\n")

# Create heatmap
plt.figure(figsize=(16, 14))
sns.heatmap(corr_matrix, 
            annot=True,  # Show correlation values
            fmt='.2f',   # Format to 2 decimal places
            cmap='RdYlGn',  # Red-Yellow-Green color scheme
            center=0,    # Center color at 0
            vmin=-1, vmax=1,  # Correlation range
            square=True,  # Square cells
            linewidths=0.5,
            cbar_kws={"shrink": 0.8})

plt.title('Cross-Asset Correlation Matrix\n(Stocks, Crypto, Forex)', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Assets', fontsize=12, fontweight='bold')
plt.ylabel('Assets', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("✅ Heatmap created!")

# Key insights
print("\n💡 KEY INSIGHTS:\n")

# 1. Strongest positive correlation
strongest_pos = corr_pairs_df.nlargest(1, 'correlation').iloc[0]
print(f"1️⃣ Strongest positive correlation:")
print(f"   {strongest_pos['asset1']} ({strongest_pos['asset1_type']}) <-> {strongest_pos['asset2']} ({strongest_pos['asset2_type']})")
print(f"   Correlation: {strongest_pos['correlation']:.4f}")
print(f"   ➡️ These assets move together closely\n")

# 2. Strongest negative correlation
strongest_neg = corr_pairs_df.nsmallest(1, 'correlation').iloc[0]
print(f"2️⃣ Strongest negative correlation:")
print(f"   {strongest_neg['asset1']} ({strongest_neg['asset1_type']}) <-> {strongest_neg['asset2']} ({strongest_neg['asset2_type']})")
print(f"   Correlation: {strongest_neg['correlation']:.4f}")
print(f"   ➡️ These assets move in opposite directions (good for hedging)\n")

# 3. Cross-asset insights
stock_crypto = corr_pairs_df[
    ((corr_pairs_df['asset1_type'] == 'STOCK') & (corr_pairs_df['asset2_type'] == 'CRYPTO')) |
    ((corr_pairs_df['asset1_type'] == 'CRYPTO') & (corr_pairs_df['asset2_type'] == 'STOCK'))
]
if len(stock_crypto) > 0:
    print(f"3️⃣ Stock-Crypto relationship:")
    print(f"   Average correlation: {stock_crypto['correlation'].mean():.4f}")
    print(f"   Range: {stock_crypto['correlation'].min():.4f} to {stock_crypto['correlation'].max():.4f}")
    print(f"   ➡️ {'Weak' if abs(stock_crypto['correlation'].mean()) < 0.3 else 'Moderate' if abs(stock_crypto['correlation'].mean()) < 0.7 else 'Strong'} relationship\n")

stock_forex = corr_pairs_df[
    ((corr_pairs_df['asset1_type'] == 'STOCK') & (corr_pairs_df['asset2_type'] == 'FOREX')) |
    ((corr_pairs_df['asset1_type'] == 'FOREX') & (corr_pairs_df['asset2_type'] == 'STOCK'))
]
if len(stock_forex) > 0:
    print(f"4️⃣ Stock-Forex relationship:")
    print(f"   Average correlation: {stock_forex['correlation'].mean():.4f}")
    print(f"   Range: {stock_forex['correlation'].min():.4f} to {stock_forex['correlation'].max():.4f}")
    print(f"   ➡️ Currency movements {'weakly' if abs(stock_forex['correlation'].mean()) < 0.3 else 'moderately' if abs(stock_forex['correlation'].mean()) < 0.7 else 'strongly'} correlate with stocks\n")

print("✅ Analysis complete!")